In [3]:
import pandas as pd
from pathlib import Path
import zipfile, io

def _fetch_data() -> pd.DataFrame:
    #content = requests.get("https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip", verify=False).content
    content = Path("data/bike.zip").read_bytes()
    with zipfile.ZipFile(io.BytesIO(content)) as arc:
        raw_data = pd.read_csv(arc.open("hour.csv"), header=0, sep=',', parse_dates=['dteday']) 
    return raw_data

df=_fetch_data

In [1]:
import time
start = time.perf_counter()

In [14]:
end = time.perf_counter()
duration = end - start
f"{duration:.3f}"

In [18]:
from pydantic import BaseModel, Field
from datetime import datetime , date, time as dt_time
import time
from typing import List, Any, Dict

class Period(BaseModel):
    start: datetime
    end: datetime

class Periods(BaseModel):
    #key et value
    dict: Dict[str, Period]

z=Periods(dict={
        "jan11"  : Period(start="2011-01-01 00:00:00", end="2011-01-28 23:59:59"),
        "feb11"  : Period(start="2011-02-01 00:00:00", end="2011-02-28 23:59:59"),
        "week 1" : Period(start="2011-01-29 00:00:00", end="2011-02-07 23:59:59"),
        "week 2" : Period(start="2011-02-07 00:00:00", end="2011-02-14 23:59:59"),
        "week 3" : Period(start="2011-02-15 00:00:00", end="2011-02-21 23:59:59"),
    })


In [1]:
from evidently import Report, Dataset, DataDefinition, Regression
from evidently.metrics import MAE, RMSE, R2Score
from evidently.presets import DataDriftPreset, RegressionPreset
#import logging
#from contextlib import asynccontextmanager
#from fastapi import FastAPI, HTTPException, Response, Request
#from pydantic import BaseModel, Field
#from dataclasses import dataclass, field
#import requests
#from prometheus_client import Counter, Histogram, generate_latest, CollectorRegistry, Gauge
from sklearn.ensemble import RandomForestRegressor
#from sklearn import model_selection
import zipfile
import pandas as pd
from datetime import datetime , date, time as dt_time
import time
import io, typing
from typing import Optional
import joblib, pickle
from pathlib import Path
from typing import List, Any, Dict
import asyncio
import sys, json

In [2]:
calendar = { 
      "jan11"  : ['2011-01-01 00:00:00' , '2011-01-31 23:59:59'] 
    , "feb11"  : ['2011-02-01 00:00:00' , '2011-02-28 23:59:59']          
    , "week 1" : ['2011-02-01 00:00:00' , '2011-02-08 23:59:59']
    , "week 2" : ['2011-02-09 00:00:00' , '2011-02-15 23:59:59']
    , "week 3" : ['2011-02-16 00:00:00' , '2011-02-22 23:59:59']
}
my_data_loc  :Path   = Path("./data/bike.zip")
DTEDAY_COL_NAME = 'dteday'
def _fetch_data() -> pd.DataFrame:
    """Fetches the bike sharing dataset and returns a DataFrame."""
    print("Fetching data from UCI archive...")
    try:
        #content = requests.get(DATASET_URL, verify=False, timeout=60).content
        content = Path(my_data_loc).read_bytes()
        with zipfile.ZipFile(io.BytesIO(content)) as z:
            df = pd.read_csv(z.open("hour.csv"), header=0, sep=',', parse_dates=[DTEDAY_COL_NAME])
        print("Data fetched successfully.")
        return df
    
    except Exception as e:
        print(f"Error processing fetched data: {e}")
        

def _process_data(raw_data: pd.DataFrame) -> pd.DataFrame:
    """Processes raw data, setting a DatetimeIndex as in the exam script."""
    print("Processing raw data...")
    raw_data['hr'] = raw_data['hr'].astype(int)
    raw_data.index = raw_data.apply(
        lambda row: datetime.combine(row[DTEDAY_COL_NAME].date(), dt_time(row.hr)),
        axis=1
    )
    raw_data = raw_data.sort_index()
    print("Data processed successfully.")
    return raw_data

# Gateway pour l'entrainement du modèle
def get_subset(raw_data: pd.DataFrame,period):
    # Reference and current data split
    return raw_data.loc[period[0]:period[1]]

prediction="prediction"
target="cnt"
numerical_features   = NUM_FEATS = ['temp', 'atemp', 'hum', 'windspeed', 'mnth', 'hr', 'weekday']
categorical_features = CAT_FEATS = ['season', 'holiday', 'workingday', 'weathersit']
all_model_feats      = ALL_MODEL_FEATS = NUM_FEATS + CAT_FEATS
my_model_loc :Path   = Path("./models/RFRegressor.pkl")
with open(my_model_loc, "rb") as f:
    app_state_RFRegressor = pickle.load(f)

/home/ubuntu/PromGraf-MLOps-Exam-Student/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ubuntu/PromGraf-MLOps-Exam-Student/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
def get_by_name(results, name):
    for k, v in results.items():
        if v.display_name == name:
            return v
    return None


def evaluate():
    start_time = time.perf_counter() # Début du timer pour la durée de la requête
    status_code = "200"

    raw_data = _process_data(_fetch_data())
    # 1. Convertir le payload en DataFrame
    #logger.info(aPeriodName.name)

    current_df = get_subset(raw_data,calendar["feb11"])
    #logger.info(current_df.shape)
    nbitems=current_df.shape[0]
    #logger.info(nbitems)

    # 2. Prédictions avec le modèle gelé
    current_df[prediction] = app_state_RFRegressor.predict(current_df[numerical_features + categorical_features])

    # 3. Dataset Evidently wrappé
    data_definition = DataDefinition(
        regression=[Regression(target=target, prediction=prediction)],
        numerical_columns=numerical_features + [target, prediction]
    )

    #benchmark
    raw_data = _process_data(_fetch_data())
    reference_jan11 = get_subset(raw_data,calendar["jan11"])
    reference_jan11[prediction] = app_state_RFRegressor.predict(reference_jan11[numerical_features + categorical_features])
    logger_info=print
    logger_info(current_df.shape)
    logger_info(reference_jan11.shape)
    
    logger_info(current_df.head())

    logger_info(reference_jan11.head())

    logger_info(reference_jan11[[target, prediction]].isnull().sum())
    logger_info(current_df[[target, prediction]].isnull().sum())
    logger_info(reference_jan11[[target, prediction]].describe())

    logger_info(current_df[[target, prediction]].describe())

    reference_dataset = Dataset.from_pandas(reference_jan11, data_definition=data_definition)
    current_dataset   = Dataset.from_pandas(current_df     , data_definition=data_definition)

    # 4. Rapport Evidently
    regpr, rmse, mae, r2score,datadrift = RegressionPreset(), RMSE(), MAE(), R2Score(), DataDriftPreset()
    logger_info([rmse, mae, r2score,datadrift])
    #report = Report(metrics=[RegressionPreset(), rmse, mae, r2score,datadrift ])
    report = Report(metrics=[regpr, rmse, mae, r2score,datadrift ])
    snapshot = report.run(reference_data=reference_dataset, current_data=current_dataset)

    # 5. Extraire les métriques de Evidently
    # logger.info(snapshot)
    results = snapshot.metric_results
    #results_dict = snapshot.as_dict()
    logger_info("LIST OF METRICS")
    logger_info(type(snapshot))
    #logger.info(results_dict["metrics"])
    logger_info([m for m in dir(snapshot) if not m.startswith("_")])

    results = snapshot.metric_results

    # AVOIR un affichage systématique des clefs de métrics obtenues et des labels associés
    # objectif, savoir si un accés par clés ou par labels est approprié pour différencier les métriques du report
    # récupération du tableau des cléfs de métric
    resuKeys=list(results.keys())
    # affichage du display name
    for z in [(resuKeys[k],results[resuKeys[k]].display_name) for k in range(len(resuKeys))]:
        logger_info(z)
    #attention, je donne cette méthode, mais dans le debugger de vscode les quatre premières métrics ne sont jamais "display"

    # logger.info(results)
    rmse  = get_by_name(results, "RMSE")
    mae   = get_by_name(results, "Mean Absolute Error")
    r2    = get_by_name(results, "R2 Score")
    drift = get_by_name(results, "Count of Drifted Columns")

    v_rmse    = rmse.value
    v_mae     = mae.mean.value
    v_mae_std = mae.std.value
    v_r2      = r2.value
    v_drift_count = drift.count.value
    v_drift_share = drift.share.value

    # 6. Mettre à jour Prometheus
    labels = {"endpoint": "/evaluate", "method": "POST", "status_code": "200"}
    #PROM_model_rmse_score.labels(**labels).set(rmse)
    #PROM_model_mae_score.labels(**labels).set(mae)
    #PROM_model_r2_score.labels(**labels).set(r2)

    end_time = time.perf_counter()
    # Durée de la requête
    duration = end_time - start_time
    #return EvaluationReportOutput(message=f"evaluation terminated in {duration:.3f}", rmse=v_rmse, mae=v_mae, r2=v_r2, drift_detected=v_drift, evaluated_items= nbitems)
    return {"rmse":v_rmse, "mae":v_mae, "r2":v_r2, "drift_detected":v_drift_count}
evaluate()

Fetching data from UCI archive...
Data fetched successfully.
Processing raw data...
Data processed successfully.
Fetching data from UCI archive...
Data fetched successfully.
Processing raw data...
Data processed successfully.
(649, 18)
(688, 18)
                     instant     dteday  season  yr  mnth  hr  holiday  \
2011-02-01 00:00:00      689 2011-02-01       1   0     2   0        0   
2011-02-01 01:00:00      690 2011-02-01       1   0     2   1        0   
2011-02-01 02:00:00      691 2011-02-01       1   0     2   2        0   
2011-02-01 03:00:00      692 2011-02-01       1   0     2   3        0   
2011-02-01 05:00:00      693 2011-02-01       1   0     2   5        0   

                     weekday  workingday  weathersit  temp   atemp   hum  \
2011-02-01 00:00:00        2           1           2  0.16  0.1818  0.64   
2011-02-01 01:00:00        2           1           2  0.16  0.1818  0.69   
2011-02-01 02:00:00        2           1           2  0.16  0.2273  0.69   
2011-

/home/ubuntu/PromGraf-MLOps-Exam-Student/.venv/lib/python3.10/site-packages/sklearn/metrics/_regression.py:1283: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

/home/ubuntu/PromGraf-MLOps-Exam-Student/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning:

invalid value encountered in divide

/home/ubuntu/PromGraf-MLOps-Exam-Student/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning:

invalid value encountered in divide

/home/ubuntu/PromGraf-MLOps-Exam-Student/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning:

invalid value encountered in divide

/home/ubuntu/PromGraf-MLOps-Exam-Student/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning:

invalid value encountered in divide

/home/ubuntu/PromGraf-MLOps-Exam-Student/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarni

COUCOUCOUCOU
<class 'evidently.core.report.Snapshot'>
['context', 'dict', 'dump_dict', 'dumps', 'get_html_str', 'get_name', 'json', 'load', 'load_dict', 'load_model', 'loads', 'metric_results', 'render_only_fingerprint', 'report', 'run', 'save_html', 'save_json', 'set_name', 'tests_results', 'to_snapshot_model']
('eb50f621dc9ac96982148a761b11eaa7', 'Mean Error')
('5aa32fd1e3c5da1cd47cbc5f7ab4372d', 'Mean Absolute Percentage Error')
('07a6d46c093f8fd4d3fd9a8f834e0446', 'RMSE')
('388a1f90d7785053ba4b44301e310cf2', 'Mean Absolute Error')
('9c7e7b78ceb0ed9bc65fd16a8cca6237', 'R2 Score')
('d2ac9457e6d5eacc29f1a47de21d9d45', 'Absolute Max Error')
('15e89f895b482f9b84ba7274ed18a106', 'Count of Drifted Columns')
('effcb22215e0783e0a062f57fe24c942', 'Value drift for temp')
('417beabf503d3641e3623a41134f1b1c', 'Value drift for atemp')
('cdadbdc3b8f0abddd09c4291d81d9a63', 'Value drift for hum')
('cd441c0154413908a8e3dae456b6929a', 'Value drift for windspeed')
('4e4027daa585d2c119c2841acdb3d3b1', 

{'rmse': 31.533976217506833,
 'mae': 19.690169491525424,
 'r2': 0.7539871128200374,
 'drift_detected': 8.0}

In [ ]:
results